# Week 04 - Homework: EU AI Act RAG Evaluation

You will build a comprehensive RAG evaluation system for EU AI Act compliance questions using the concepts from Week 04 Lesson 01 - 03.

### What You'll Build

- **EU AI Act RAG System**: Document retrieval and answer generation for compliance questions
- **Evaluation Framework**: Multi-dimensional assessment using 4 key evaluators
- **Target Function**: Wrapper function for systematic evaluation
- **Comprehensive Testing**: Full evaluation pipeline with metrics

### Background: EU AI Act

The EU AI Act is a comprehensive regulatory framework for artificial intelligence in the European Union. It establishes rules for AI systems based on their risk levels and includes requirements for transparency, accountability, and human oversight.

>**TODO**: Complete the evaluation sections based on Lesson 03

---


## Part 1: Environment Setup and Dependencies


In [ ]:
# Install required dependencies
%pip install -U --quiet langsmith langchain langchain-openai langchain-community python-dotenv openai tiktoken pypdf requests


In [ ]:
# Import necessary libraries
import os
import requests
from typing_extensions import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import Tool
from langsmith import Client, traceable
from langsmith.evaluation import evaluate
from langsmith.schemas import Run, Example


In [ ]:
# Load environment variables
load_dotenv()

# Set up LangSmith environment variables
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = 'your_api_key'
os.environ['LANGSMITH_PROJECT'] = 'eu-ai-act-rag-evaluation'
os.environ['OPENAI_API_KEY'] = 'sk-your-api-key'

# Initialize LangSmith client
client = Client()
print("✓ Environment configured successfully!")


## Part 2: EU AI Act Document Processing

First, we'll download and process the EU AI Act document to create our knowledge base.


In [ ]:
# Load EU AI Act PDF document
def load_eu_ai_act_pdf():
    """Load the EU AI Act PDF document."""
    pdf_path = "eu_ai_act.pdf"
    
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at {pdf_path}")
        print("Please ensure the eu_ai_act.pdf file is in the current directory")
        return None
    
    # Load PDF using PyPDFLoader
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    print(f"✓ Loaded EU AI Act PDF with {len(documents)} pages")
    return documents

# Load the document
documents = load_eu_ai_act_pdf()
if documents:
    print("✓ EU AI Act PDF loaded successfully!")
else:
    print("✗ Failed to load EU AI Act PDF")


In [ ]:
# Process the document and create vector store
def create_knowledge_base(documents):
    """Create a knowledge base from the EU AI Act PDF documents."""
    
    if not documents:
        print("Error: No documents provided")
        return None
    
    # Split documents
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=1000,
        chunk_overlap=200
    )
    
    doc_splits = text_splitter.split_documents(documents)
    print(f"✓ Created {len(doc_splits)} document chunks")
    
    # Create vector store
    vectorstore = InMemoryVectorStore.from_documents(
        documents=doc_splits,
        embedding=OpenAIEmbeddings()
    )
    
    return vectorstore

# Create knowledge base
if documents:
    vectorstore = create_knowledge_base(documents)
    retriever = vectorstore.as_retriever(k=4)
    print("✓ Knowledge base created successfully!")
else:
    print("✗ Cannot create knowledge base without documents")


## Part 3: RAG Implementation

Now we'll create an agent that can reason about and answer questions regarding the EU AI Act.


In [ ]:
# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

# Create RAG tool for the agent
@traceable
def rag_search(query: str) -> str:
    """Search the EU AI Act document for relevant information."""
    try:
        # Retrieve relevant documents
        docs = retriever.invoke(query)
        
        # Combine document content
        context = "\n\n".join([doc.page_content for doc in docs])
        
        return f"Relevant information from EU AI Act:\n{context}"
    except Exception as e:
        return f"Error searching documents: {str(e)}"

# Create the RAG tool
rag_tool = Tool(
    name="eu_ai_act_search",
    description="Search the EU AI Act document for information about AI regulations, compliance requirements, and legal provisions",
    func=rag_search
)

print("✓ RAG tool created successfully!")


In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

# Creating the agent
agent_executor = create_react_agent(model=llm, tools=[rag_tool], checkpointer=InMemorySaver())

# Create the agent
print("✓ ReAct agent created successfully!")


In [ ]:
# Test the agent
@traceable
def test_agent(question: str) -> str:
    """Test the ReAct agent with a question."""
    try:
        config = {"configurable": {"thread_id": "test_thread"}}
        response = agent_executor.invoke({"messages": question}, config=config)
        return response["messages"][-1].content
    except Exception as e:
        return f"Error: {str(e)}"

# Test with a sample question
test_question = "What are the requirements for obligations of deployers of high-risk AI systems under the EU AI Act?"
print(f"Question: {test_question}")
print("\n" + "="*80)
response = test_agent(test_question)
print(f"Answer: {response}")


## Part 4: RAG Evaluation Framework

Now we'll implement comprehensive RAG evaluation using the techniques from Week 04 Lesson 02.


In [ ]:
# Create evaluation dataset for EU AI Act RAG
evaluation_examples = [
    {
        "inputs": {"question": "What are the prohibited AI practices under the EU AI Act?"},
        "metadata": {
            "category": "prohibited_practices",
        }
    },
    {
        "inputs": {"question": "What is a high-risk AI system according to the EU AI Act?"},
        "metadata": {
            "category": "definitions",
        }
    },
    {
        "inputs": {"question": "What are the requirements for providers of high-risk AI systems?"},
        "metadata": {
            "category": "compliance",
        }
    },
    {
        "inputs": {"question": "What is the conformity assessment procedure for high-risk AI systems?"},
        "metadata": {
            "category": "procedures",
        }
    },
    {
        "inputs": {"question": "What are the transparency requirements for AI systems?"},
        "metadata": {
            "category": "transparency",
        }
    }
    # TODO:Add more examples here to evaluate the agent's performance
]

print(f"✓ Created {len(evaluation_examples)} evaluation examples")


In [ ]:
# Create dataset in LangSmith
dataset_name = "eu-ai-act-rag-evaluation"

try:
    # Create dataset
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="EU AI Act RAG evaluation dataset with compliance questions and reference answers"
    )
    
    # Add examples to dataset
    client.create_examples(
        dataset_id=dataset.id,
        examples=evaluation_examples
    )
    
    print(f"✓ Dataset '{dataset_name}' created successfully!")
    print(f"Dataset ID: {dataset.id}")
    
except Exception as e:
    if "already exists" in str(e):
        # Use existing dataset
        dataset = client.read_dataset(dataset_name=dataset_name)
        print(f"✓ Using existing dataset: {dataset.id}")
    else:
        print(f"Error creating dataset: {e}")


---

## Part 5: Individual Evaluators Implementation

### TODO: Copy the 4 evaluators from Lesson 03
1. **Correctness Evaluator**: Copy the schema, instructions, and function from Lesson 03
2. **Relevance Evaluator**: Copy the schema, instructions, and function from Lesson 03  
3. **Groundedness Evaluator**: Copy the schema, instructions, and function from Lesson 03
4. **Retrieval Relevance Evaluator**: Copy the schema, instructions, and function from Lesson 03

These four evaluation techniques assess different dimensions of RAG system performance:
- **Correctness**: Response vs reference answer
- **Relevance**: Response vs input question  
- **Groundedness**: Response vs retrieved documents
- **Retrieval Relevance**: Retrieved documents vs input question

In [ ]:
# TODO: Copy the Correctness Evaluator from Lesson 03
# 1. Copy the CorrectnessGrade TypedDict schema
# 2. Copy the correctness_instructions prompt
# 3. Copy the grader_llm initialization
# 4. Copy the correctness function
# 5. Adapt the prompt for EU AI Act compliance questions


In [ ]:
# TODO: Copy the Relevance Evaluator from Lesson 03
# 1. Copy the RelevanceGrade TypedDict schema
# 2. Copy the relevance_instructions prompt
# 3. Copy the relevance_llm initialization
# 4. Copy the relevance function
# 5. Adapt the prompt for EU AI Act compliance questions


In [ ]:
# TODO: Copy the Groundedness Evaluator from Lesson 03
# 1. Copy the GroundedGrade TypedDict schema
# 2. Copy the grounded_instructions prompt
# 3. Copy the grounded_llm initialization
# 4. Copy the groundedness function
# 5. Adapt the prompt for EU AI Act compliance questions


In [ ]:
# TODO: Copy the Retrieval Relevance Evaluator from Lesson 03
# 1. Copy the RetrievalRelevanceGrade TypedDict schema
# 2. Copy the retrieval_relevance_instructions prompt
# 3. Copy the retrieval_relevance_llm initialization
# 4. Copy the retrieval_relevance function
# 5. Adapt the prompt for EU AI Act compliance questions


---

## Part 6: Testing and Evaluation Execution

### TODO: Test evaluators and run comprehensive evaluation
1. **Individual Testing**: Test each evaluator with sample questions
2. **Full Evaluation**: Run comprehensive evaluation using all evaluators
3. **Results Analysis**: Interpret and analyze the evaluation results


In [ ]:
# TODO: Copy AND ADAPT the target function from Lesson 03
# TODO: YOU NEED TO ADAPT THE TARGET FUNCTION TO USE THE EU AI ACT RAG SYSTEM

# Target function for React Agent Executor evaluation
# Create a function that takes dataset inputs and returns RAG outputs
# The function should wrap your RAG assistant and return both answer and documents

In [ ]:
# TODO: Test individual evaluators (copy from Lesson 03)
# Test each evaluator with sample questions to ensure they work correctly
# Test correctness, relevance, groundedness, and retrieval relevance


In [ ]:
# TODO: Adapt the evaluation code
# Modify the evaluate() function call to use your evaluators
# Include all 4 evaluators: [correctness, relevance, groundedness, retrieval_relevance]
# Make sure your target_function is properly implemented


---

## Part 7: Results Analysis and Conclusion

### TODO: Analyze your evaluation results
1. **Review LangSmith Results**: Check the evaluation results in the LangSmith UI
2. **Interpret Scores**: Analyze correctness, relevance, groundedness, and retrieval relevance scores
3. **Identify Issues**: Look for patterns in failed evaluations
4. **Document Insights**: Write down key findings and areas for improvement

### Key Takeaways
- **Multi-dimensional Evaluation**: Use multiple evaluators for comprehensive RAG assessment
- **LLM-as-Judge**: Leverage advanced models for sophisticated evaluation
- **Domain Adaptation**: Tailor evaluation criteria to your specific domain (EU AI Act)
- **Continuous Improvement**: Use evaluation insights to enhance your RAG system

**Remember**: RAG evaluation is an ongoing process that requires continuous refinement based on real-world performance and evolving requirements!
